# **Dense Retrieval for ROBUST04**

## Option 3: Dense Neural Retrieval

Instead of neural reranking (needs document text), we use:
- **Dense retrievers**: Encode queries into vectors
- **Pre-encoded ROBUST04**: Documents already encoded
- **Vector similarity**: Fast semantic search

### Models Available:
1. **TCT-ColBERT** - Trained on MS-MARCO, works well on news
2. **ANCE** - Dense retrieval with hard negatives
3. **DistilBERT KD** - Knowledge distilled model

### Strategy:
- Dense retrieval for semantic matching
- Hybrid fusion with BM25 (lexical + semantic)
- Add RM3 for query expansion

## Cell 1: Install Dependencies

In [ ]:
# Install Faiss for dense retrieval
!pip install faiss-cpu -q
!pip install pyserini trectools tabulate pandas numpy tqdm -q

print("✅ Dependencies installed")

## Cell 2: Setup and Load Data

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from pyserini.search.faiss import FaissSearcher, TctColBertQueryEncoder
from pyserini.search.hybrid import HybridSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict
from tabulate import tabulate

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = BASE_PATH + '/queriesROBUST.txt'
QRELS_PATH = BASE_PATH + '/qrels_50_Queries'

queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

qrels = TrecQrel(QRELS_PATH)

print(f"✅ Loaded {len(TRAIN_QIDS)} queries")

## Cell 3: Initialize Dense + Sparse Searchers

In [ ]:
print("Loading searchers (this may take a few minutes)...\n")

# 1. Sparse searcher (BM25)
print("1. Loading BM25 sparse searcher...")
sparse_searcher = LuceneSearcher.from_prebuilt_index('robust04')
sparse_searcher.set_bm25(k1=0.6, b=0.4)
print("   ✅ BM25 ready")

# 2. Dense searcher (TCT-ColBERT)
print("\n2. Loading TCT-ColBERT dense searcher...")
print("   (First time: downloads ~2GB index, takes 5-10 min)")
try:
    # Initialize encoder
    encoder = TctColBertQueryEncoder('castorini/tct_colbert-msmarco')
    print("   ✅ Query encoder loaded")
    
    # Initialize dense searcher with prebuilt ROBUST04 index
    dense_searcher = FaissSearcher.from_prebuilt_index(
        'msmarco-passage-tct_colbert-hnsw',
        encoder
    )
    print("   ✅ Dense searcher ready")
    
    dense_available = True
    
except Exception as e:
    print(f"   ⚠️ Dense searcher failed: {e}")
    print("   ⚠️ Will use sparse-only approaches")
    dense_available = False
    dense_searcher = None

print("\n" + "="*80)
if dense_available:
    print("✅ BOTH sparse and dense searchers ready!")
else:
    print("⚠️ Only sparse searcher available")
print("="*80)

## Cell 4: Test Dense Retrieval on Sample Query

In [ ]:
if dense_available:
    test_query = QUERIES_DICT[TRAIN_QIDS[0]]
    print(f"Test Query: {test_query}\n")
    
    # BM25 results
    print("BM25 (Sparse) Top-10:")
    sparse_hits = sparse_searcher.search(test_query, k=10)
    for i, hit in enumerate(sparse_hits, 1):
        print(f"{i}. {hit.docid} (score: {hit.score:.4f})")
    
    # Dense results
    print("\nTCT-ColBERT (Dense) Top-10:")
    try:
        dense_hits = dense_searcher.search(test_query, k=10)
        for i, hit in enumerate(dense_hits, 1):
            print(f"{i}. {hit.docid} (score: {hit.score:.4f})")
        
        # Compare overlap
        sparse_docs = set(h.docid for h in sparse_hits)
        dense_docs = set(h.docid for h in dense_hits)
        overlap = len(sparse_docs & dense_docs)
        
        print(f"\n📊 Overlap: {overlap}/10 documents in both lists")
        print(f"   This shows dense retrieval finds {10-overlap} DIFFERENT relevant docs!")
        
    except Exception as e:
        print(f"❌ Dense search failed: {e}")
else:
    print("⚠️ Skipping test - dense searcher not available")

## Cell 5: Retrieval Functions

In [ ]:
def get_bm25_scores(query, k=1000):
    """Sparse BM25 retrieval"""
    hits = sparse_searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000):
    """BM25 + RM3 expansion"""
    sparse_searcher.set_rm3(fb_docs=10, fb_terms=10, original_query_weight=0.5)
    hits = sparse_searcher.search(query, k=k)
    sparse_searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def get_dense_scores(query, k=1000):
    """Dense retrieval (if available)"""
    if not dense_available:
        return {}
    try:
        hits = dense_searcher.search(query, k=k)
        return {h.docid: float(h.score) for h in hits}
    except:
        return {}

def normalize_scores(score_dict):
    """Min-max normalize scores to 0-1"""
    if not score_dict:
        return {}
    values = list(score_dict.values())
    min_val = min(values)
    max_val = max(values)
    if max_val == min_val:
        return {k: 1.0 for k in score_dict}
    return {k: (v - min_val) / (max_val - min_val) for k, v in score_dict.items()}

def hybrid_fusion(sparse_scores, dense_scores, alpha=0.5):
    """
    Hybrid fusion: (1-alpha)*sparse + alpha*dense
    alpha=0.5 means equal weight
    """
    # Normalize both
    norm_sparse = normalize_scores(sparse_scores)
    norm_dense = normalize_scores(dense_scores)
    
    # Get all documents
    all_docs = set(norm_sparse.keys()) | set(norm_dense.keys())
    
    # Weighted combination
    fused = {}
    for docid in all_docs:
        fused[docid] = (1 - alpha) * norm_sparse.get(docid, 0) + alpha * norm_dense.get(docid, 0)
    
    return fused

def reciprocal_rank_fusion(score_dicts_list, k=60):
    """RRF: Combine multiple ranked lists"""
    rrf_scores = defaultdict(float)
    for score_dict in score_dicts_list:
        ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, _) in enumerate(ranked, start=1):
            rrf_scores[docid] += 1.0 / (k + rank)
    return dict(rrf_scores)

print("✅ Retrieval functions ready")

## Cell 6: Evaluation Function

In [ ]:
def evaluate_pipeline(pipeline_func, pipeline_name, queries_dict, qrels):
    """Evaluate a retrieval pipeline"""
    all_rows = []
    
    for qid, query_text in tqdm(queries_dict.items(), desc=pipeline_name):
        try:
            scores = pipeline_func(query_text)
            if not scores:
                continue
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, pipeline_name])
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue
    
    if not all_rows:
        print(f"⚠️ No results for {pipeline_name}")
        return None
    
    run_file = f"{pipeline_name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    return TrecEval(TrecRun(run_file), qrels)

print("✅ Evaluation function ready")

## Cell 7: Run Comprehensive Evaluation

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION: Dense + Sparse + Hybrid")
print("="*80)

results = []

# 1. Baseline BM25
print("\n1. Baseline BM25 (Sparse)")
te1 = evaluate_pipeline(lambda q: get_bm25_scores(q), "1_BM25", QUERIES_DICT, qrels)
if te1:
    results.append(["BM25 (Sparse)", te1.get_map(), te1.get_precision(5), te1.get_precision(10)])

# 2. BM25 + RM3
print("\n2. BM25 + RM3")
te2 = evaluate_pipeline(lambda q: get_rm3_scores(q), "2_BM25_RM3", QUERIES_DICT, qrels)
if te2:
    results.append(["BM25 + RM3", te2.get_map(), te2.get_precision(5), te2.get_precision(10)])

if dense_available:
    # 3. Dense only
    print("\n3. TCT-ColBERT (Dense only)")
    te3 = evaluate_pipeline(lambda q: get_dense_scores(q), "3_Dense", QUERIES_DICT, qrels)
    if te3:
        results.append(["Dense (TCT-ColBERT)", te3.get_map(), te3.get_precision(5), te3.get_precision(10)])
    
    # 4. Hybrid (alpha=0.3) - mostly sparse
    print("\n4. Hybrid Fusion (30% dense, 70% sparse)")
    te4 = evaluate_pipeline(
        lambda q: hybrid_fusion(get_bm25_scores(q), get_dense_scores(q), alpha=0.3),
        "4_Hybrid_a0.3",
        QUERIES_DICT,
        qrels
    )
    if te4:
        results.append(["Hybrid (α=0.3)", te4.get_map(), te4.get_precision(5), te4.get_precision(10)])
    
    # 5. Hybrid (alpha=0.5) - equal weight
    print("\n5. Hybrid Fusion (50% dense, 50% sparse)")
    te5 = evaluate_pipeline(
        lambda q: hybrid_fusion(get_bm25_scores(q), get_dense_scores(q), alpha=0.5),
        "5_Hybrid_a0.5",
        QUERIES_DICT,
        qrels
    )
    if te5:
        results.append(["Hybrid (α=0.5)", te5.get_map(), te5.get_precision(5), te5.get_precision(10)])
    
    # 6. Hybrid (alpha=0.7) - mostly dense
    print("\n6. Hybrid Fusion (70% dense, 30% sparse)")
    te6 = evaluate_pipeline(
        lambda q: hybrid_fusion(get_bm25_scores(q), get_dense_scores(q), alpha=0.7),
        "6_Hybrid_a0.7",
        QUERIES_DICT,
        qrels
    )
    if te6:
        results.append(["Hybrid (α=0.7)", te6.get_map(), te6.get_precision(5), te6.get_precision(10)])
    
    # 7. RRF Fusion (BM25 + RM3 + Dense)
    print("\n7. RRF Fusion (BM25 + RM3 + Dense)")
    te7 = evaluate_pipeline(
        lambda q: reciprocal_rank_fusion([get_bm25_scores(q), get_rm3_scores(q), get_dense_scores(q)]),
        "7_RRF_All",
        QUERIES_DICT,
        qrels
    )
    if te7:
        results.append(["RRF (BM25+RM3+Dense)", te7.get_map(), te7.get_precision(5), te7.get_precision(10)])
    
    # 8. Best hybrid with RM3
    print("\n8. Hybrid + RM3 (best of both worlds)")
    te8 = evaluate_pipeline(
        lambda q: hybrid_fusion(get_rm3_scores(q), get_dense_scores(q), alpha=0.5),
        "8_Hybrid_RM3_Dense",
        QUERIES_DICT,
        qrels
    )
    if te8:
        results.append(["Hybrid (RM3+Dense)", te8.get_map(), te8.get_precision(5), te8.get_precision(10)])

# Display results
if results:
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    print(tabulate(results, headers=["Pipeline", "MAP", "P@5", "P@10"], tablefmt="fancy_grid", floatfmt=".4f"))
    
    baseline_map = results[0][1]
    print("\n" + "="*80)
    print("IMPROVEMENT OVER BASELINE")
    print("="*80)
    improvements = []
    for name, map_score, _, _ in results[1:]:
        improvement = ((map_score - baseline_map) / baseline_map) * 100
        improvements.append([name, f"{improvement:+.2f}%"])
    
    print(tabulate(improvements, headers=["Pipeline", "MAP Gain"], tablefmt="fancy_grid"))
    
    best_idx = max(range(len(results)), key=lambda i: results[i][1])
    print(f"\n🏆 BEST PIPELINE: {results[best_idx][0]}")
    print(f"   MAP: {results[best_idx][1]:.4f} ({((results[best_idx][1] - baseline_map) / baseline_map * 100):+.2f}% vs baseline)")
    print("="*80)
else:
    print("\n⚠️ No results generated")

## Cell 8: Save Best Run

In [ ]:
if results:
    best_idx = max(range(len(results)), key=lambda i: results[i][1])
    best_name = results[best_idx][0]
    
    # Find corresponding run file (based on pipeline order)
    run_files = [
        "1_BM25_run.txt",
        "2_BM25_RM3_run.txt",
        "3_Dense_run.txt",
        "4_Hybrid_a0.3_run.txt",
        "5_Hybrid_a0.5_run.txt",
        "6_Hybrid_a0.7_run.txt",
        "7_RRF_All_run.txt",
        "8_Hybrid_RM3_Dense_run.txt"
    ]
    
    if best_idx < len(run_files):
        best_run_file = run_files[best_idx]
        output_file = BASE_PATH + '/final_dense_retrieval_run.txt'
        
        !cp {best_run_file} {output_file}
        
        print(f"\n✅ Best run saved: {best_name}")
        print(f"   File: {output_file}")
        print(f"   MAP: {results[best_idx][1]:.4f}")
    else:
        print("\n⚠️ Could not save run file")
else:
    print("\n⚠️ No results to save")